# Fraud Detection Project

## Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set(style='whitegrid')

print('Libraries loaded successfully')

## Load the Dataset

In [ ]:
df = pd.read_csv('AIML Dataset.csv')

print('Dataset shape:', df.shape)
print('Rows:', df.shape[0])
print('Columns:', df.shape[1])

df.head()

## Basic Data Exploration

In [ ]:
df.info()

In [ ]:
print('Missing values in each column:')
print(df.isnull().sum())

In [ ]:
print('Fraud vs Normal transaction count:')
print(df['isFraud'].value_counts())
print()
print('As percentage:')
print(df['isFraud'].value_counts(normalize=True) * 100)

## Visualizations

In [ ]:
df['isFraud'].value_counts().plot(
    kind='bar',
    color=['steelblue', 'salmon'],
    title='Fraud vs Normal Transactions'
)
plt.xlabel('0 = Normal, 1 = Fraud')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
df['type'].value_counts().plot(
    kind='bar',
    color='skyblue',
    title='Transaction Types'
)
plt.xlabel('Transaction Type')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
fraud_by_type = df.groupby('type')['isFraud'].mean().sort_values(ascending=False)

fraud_by_type.plot(
    kind='bar',
    color='salmon',
    title='Fraud Rate by Transaction Type'
)
plt.xlabel('Transaction Type')
plt.ylabel('Fraud Rate (0 to 1)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
sns.histplot(
    np.log1p(df['amount']),
    bins=80,
    kde=True,
    color='green'
)
plt.title('Transaction Amount Distribution (log scale)')
plt.xlabel('log(Amount + 1)')
plt.tight_layout()
plt.show()

## Feature Engineering

In [ ]:
df['balanceDiffOrig'] = df['oldbalanceOrg'] - df['newbalanceOrig']
df['balanceDiffDest'] = df['newbalanceDest'] - df['oldbalanceDest']

print('New features added:')
df[['balanceDiffOrig', 'balanceDiffDest']].head()

## Prepare Data for Model

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

df_model = df.drop(columns=['nameOrig', 'nameDest', 'isFlaggedFraud'])

X = df_model.drop(columns=['isFraud'])
y = df_model['isFraud']

print('Features (X) shape:', X.shape)
print('Target (y) shape:', y.shape)
print()
print('Feature columns:', list(X.columns))

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

print('Training set size:', X_train.shape[0])
print('Test set size:', X_test.shape[0])

## Build and Train the Model

In [ ]:
from sklearn.ensemble import RandomForestClassifier

numeric_cols = [
    'step', 'amount',
    'oldbalanceOrg', 'newbalanceOrig',
    'oldbalanceDest', 'newbalanceDest',
    'balanceDiffOrig', 'balanceDiffDest'
]
categorical_cols = ['type']

preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numeric_cols),
    ('cat', OneHotEncoder(drop='first'), categorical_cols)
])

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        n_estimators=100,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    ))
])

print('Training the model... this may take a minute')
pipeline.fit(X_train, y_train)
print('Training complete')

## Evaluate the Model

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

y_pred = pipeline.predict(X_test)
y_proba = pipeline.predict_proba(X_test)[:, 1]

print('Classification Report:')
print(classification_report(y_test, y_pred, target_names=['Normal', 'Fraud']))

print('ROC-AUC Score:', round(roc_auc_score(y_test, y_proba), 4))

In [ ]:
cm = confusion_matrix(y_test, y_pred)

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=['Normal', 'Fraud'],
    yticklabels=['Normal', 'Fraud']
)
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

print('True Negatives  (Normal predicted Normal):', cm[0][0])
print('False Positives (Normal predicted Fraud) :', cm[0][1])
print('False Negatives (Fraud predicted Normal) :', cm[1][0])
print('True Positives  (Fraud predicted Fraud)  :', cm[1][1])

In [ ]:
feature_names = (
    numeric_cols +
    list(pipeline.named_steps['preprocessor']
         .named_transformers_['cat']
         .get_feature_names_out(categorical_cols))
)

importances = pipeline.named_steps['classifier'].feature_importances_

feat_df = pd.DataFrame({
    'feature': feature_names,
    'importance': importances
}).sort_values('importance', ascending=False)

plt.figure(figsize=(8, 5))
sns.barplot(data=feat_df, x='importance', y='feature', color='steelblue')
plt.title('Feature Importance')
plt.tight_layout()
plt.show()

## Save the Model

In [ ]:
import joblib

joblib.dump(pipeline, 'Fraud_detection_pipeline.pkl')

print('Model saved as Fraud_detection_pipeline.pkl')